In [1]:
!pip install pandas
!pip install sqlalchemy

In [2]:
import pandas as pd
from sqlalchemy import create_engine

In [3]:
#Why not write import sqlalchemy?
#You could — but then every time you use create_engine you'd have to write sqlalchemy.create_engine(). By writing from sqlalchemy import create_engine you can just write create_engine() directly. Less typing, more readable.

In [4]:
connect= create_engine ("sqlite:///C:/Users/Chheten/OneDrive - The City University of New York/Spring 2026/Projects/credit_risk.db")


In [14]:
KEEP_COLS = [
    'loan_amnt', 'int_rate', 'grade', 'emp_length',
    'annual_inc', 'loan_status', 'dti', 'fico_range_low',
    'fico_range_high', 'addr_state', 'purpose', 'delinq_2yrs'
]
# Only load the columns we need

In [19]:
df = pd.read_csv('accepted_2007_to_2018Q4.csv',low_memory=False) #read the whole file first then figure out the data type rather than reading it chunk by chunk to figure out.

In [31]:
# Delete the last 2 junk rows from the bottom
df = df[:-2]
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
columns_to_keep = [
    'id', 'loan_amnt', 'int_rate', 'grade', 'emp_length',
    'annual_inc', 'loan_status', 'dti', 'fico_range_low',
    'fico_range_high', 'addr_state', 'purpose', 'delinq_2yrs'
]
# Create a smaller dataframe with just these columns
borrowers_df = df[columns_to_keep].copy()
#.copy() — creates a separate copy so changes to borrowers_df don't affect the original df.
borrowers_df.head(20)

,id,loan_amnt,int_rate,grade,emp_length,annual_inc,loan_status,dti,fico_range_low,fico_range_high,addr_state,purpose,delinq_2yrs
0,68407277,3600.0,13.99,C,10+ years,55000.0,Fully Paid,5.91,675.0,679.0,PA,debt_consolidation,0.0
1,68355089,24700.0,11.99,C,10+ years,65000.0,Fully Paid,16.06,715.0,719.0,SD,small_business,1.0
2,68341763,20000.0,10.78,B,10+ years,63000.0,Fully Paid,10.78,695.0,699.0,IL,home_improvement,0.0
3,66310712,35000.0,14.85,C,10+ years,110000.0,Current,17.06,785.0,789.0,NJ,debt_consolidation,0.0
4,68476807,10400.0,22.45,F,3 years,104433.0,Fully Paid,25.37,695.0,699.0,PA,major_purchase,1.0
5,68426831,11950.0,13.44,C,4 years,34000.0,Fully Paid,10.20,690.0,694.0,GA,debt_consolidation,0.0
6,68476668,20000.0,9.17,B,10+ years,180000.0,Fully Paid,14.67,680.0,684.0,MN,debt_consolidation,0.0
7,67275481,20000.0,8.49,B,10+ years,85000.0,Fully Paid,17.61,705.0,709.0,SC,major_purchase,1.0
8,68466926,10000.0,6.49,A,6 years,85000.0,Fully Paid,13.07,685.0,689.0,PA,credit_card,0.0
9,68616873,8000.0,11.48,B,10+ years,42000.0,Fully Paid,34.80,700.0,704.0,RI,credit_card,0.0


In [50]:
# Create a new column called 'defaulted' to store our binary target variable
borrowers_df['defaulted'] = borrowers_df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else (0 if x == 'Fully Paid' else None))
# For each loan status value, convert text to number: Charged Off=1, Fully Paid=0, everything else=None

# Print a header so we know what the next output shows
print("Loan status breakdown:")

# Count how many loans have each status (Charged Off, Fully Paid, Current, etc.) in the original text column
print(borrowers_df['loan_status'].value_counts())

# Print a blank line then header for the next output to make it easier to read
print("\nDefault flag breakdown:")

# Count how many loans are 1 (defaulted) vs 0 (fully paid) in our new numeric column
print(borrowers_df['defaulted'].value_counts())

Loan status breakdown:
loan_status
Fully Paid                                             1076751
Current                                                 878312
Charged Off                                             268558
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

Default flag breakdown:
defaulted
0.0    1076751
1.0     268558
Name: count, dtype: int64


In [58]:
# Keep only clear outcomes (remove None values)
Clean_Data=borrowers_df.dropna (subset=['defaulted'].copy())
# Count how many rows we kept
row_count=len(Clean_Data)
print(row_count)
print(Clean_Data.shape) #checking rows ad columns

1345309
(1345309, 14)


In [66]:
#SQL
Clean_Data.to_sql("borrowers", connect, if_exists='replace', index=False)
#cleaned data to our SQL database, if it exisits replace it with the new data
print("Loaded")

Loaded


In [68]:
query="SELECT COUNT(*) as total FROM borrowers"
#checking to see if data is saved in the db, naming it as total becuase here sql would use Count (*) as a column name as we didnt specifiy what column we are checking
result=pd.read_sql(query,connect) #connect from cell 4 shows where our db is.
print (result)

     total
0  1345309


In [78]:
# Create risk level logic with REALISTIC thresholds based on actual data
risk_query = """
SELECT 
    id,
    fico_range_low,
    dti,
    defaulted,
    CASE
        WHEN fico_range_low < 680 THEN 'HIGH'
        WHEN fico_range_low BETWEEN 680 AND 719 THEN 'MEDIUM'
        WHEN fico_range_low >= 720 THEN 'LOW'
        ELSE 'UNKNOWN'
    END AS risk_level
FROM borrowers
WHERE defaulted IS NOT NULL
"""

# Execute the query and get results
risk_results = pd.read_sql(risk_query, connect)

# Write the results to a new table called risk_data
risk_results.to_sql('risk_data', connect, if_exists='replace', index=False)

# Confirm it worked
print("Risk data table created")

# Show breakdown of risk levels
levels = pd.read_sql("""
    SELECT 
        risk_level,
        COUNT(*) as count,
        ROUND(AVG(defaulted) * 100, 2) as default_rate_percent
    FROM risk_data
    GROUP BY risk_level
    ORDER BY default_rate_percent DESC
""", connect)
#rounded the defaulted column to check 1,0 and convert it to % with 2 decimals.
print("\nRisk level breakdown:")
print(levels)

Risk data table created

Risk level breakdown:
  risk_level   count  default_rate_percent
0       HIGH  459479                 25.28
1     MEDIUM  607529                 19.75
2        LOW  278301                 11.63


In [84]:
#connect it back to the old database to make predicitions.
ml_data=pd.read_sql("Select * FROM borrowers WHERE defaulted IS NOT NULL", connect)
print(len(ml_data))
ml_data.head()

1345309


,id,loan_amnt,int_rate,grade,emp_length,annual_inc,loan_status,dti,fico_range_low,fico_range_high,addr_state,purpose,delinq_2yrs,defaulted
0,68407277,3600.0,13.99,C,10+ years,55000.0,Fully Paid,5.91,675.0,679.0,PA,debt_consolidation,0.0,0.0
1,68355089,24700.0,11.99,C,10+ years,65000.0,Fully Paid,16.06,715.0,719.0,SD,small_business,1.0,0.0
2,68341763,20000.0,10.78,B,10+ years,63000.0,Fully Paid,10.78,695.0,699.0,IL,home_improvement,0.0,0.0
3,68476807,10400.0,22.45,F,3 years,104433.0,Fully Paid,25.37,695.0,699.0,PA,major_purchase,1.0,0.0
4,68426831,11950.0,13.44,C,4 years,34000.0,Fully Paid,10.20,690.0,694.0,GA,debt_consolidation,0.0,0.0


In [88]:
#selecting the columns I want to use as variables
Selected_Columns= ['loan_amnt', 'int_rate', 'fico_range_low', 'dti', 'annual_inc', 'delinq_2yrs'] 
Selected_df=ml_data[Selected_Columns] #new df
Predict=ml_data['defaulted']
print(Selected_df.shape)#checking records we have
print(len(Predict))
Selected_df.head()

(1345309, 6)
1345309


,loan_amnt,int_rate,fico_range_low,dti,annual_inc,delinq_2yrs
0,3600.0,13.99,675.0,5.91,55000.0,0.0
1,24700.0,11.99,715.0,16.06,65000.0,1.0
2,20000.0,10.78,695.0,10.78,63000.0,0.0
3,10400.0,22.45,695.0,25.37,104433.0,1.0
4,11950.0,13.44,690.0,10.20,34000.0,0.0


In [90]:
# Check for missing values in our features
missing_values = Selected_df.isnull().sum()

# Display which columns have missing data
print("Missing values per column:")
print(missing_values)

# Total missing
total_missing = missing_values.sum()
print(f"\nTotal missing values: {total_missing:,}")

Missing values per column:
loan_amnt           0
int_rate            0
fico_range_low      0
dti               374
annual_inc          0
delinq_2yrs         0
dtype: int64

Total missing values: 374


In [102]:
# Check for missing values before we start
missing= Selected_df.isnull().sum()
print(missing)
total_missing=missing.sum() #total missing from all columns
print("total missing:",total_missing)

loan_amnt           0
int_rate            0
fico_range_low      0
dti               374
annual_inc          0
delinq_2yrs         0
dtype: int64
total missing: 374


In [108]:
#removing the null values for prediciton
Selected_df_clean= Selected_df.dropna()
Predict_clean=Predict [Selected_df.dropna().index]
print(len(Selected_df))
print(len(Selected_df_clean))

1345309
1344935


In [136]:
from sklearn.model_selection import train_test_split #spliting data into training and test, from ML library
from sklearn.linear_model import LogisticRegression #logit regression is bascially yes or no (1 or 0)
from sklearn.metrics import roc_auc_score # to see our accuracy

# Split data into test and train data
X_train, X_test, y_train, y_test = train_test_split(Selected_df_clean, Predict_clean, test_size=0.2, random_state=25) #radnom state so it does the same suffle when i run the code so auc score dont chnage

# Train model
model = LogisticRegression(max_iter=1000) #try it 1000 times to learn the best way
model.fit(X_train, y_train) #fits the best model

# Get score
predictions = model.predict_proba(X_test)[:, 1] # model has learned the pattern and applys it to test data and we want only default column 
auc_score = roc_auc_score(y_test, predictions) # checking our predictions accuracy

print("AUC Score:",auc_score)
print ("AUC Score Percentage:", round(auc_score * 100,1),"%")

AUC Score: 0.6904754414265868
AUC Score Percentage: 69.0 %


In [137]:
# to show individual borrowers risk
all_prediction=model.predict_proba(Selected_df_clean)[:,1] #now use the same model for all the records, earlier we used it only for 20%
Predict_score=all_prediction * 100 #perecntage
Selected_df_clean ['Predict_score']= Predict_score #new column and add the values from predict_score
Selected_df_clean.head(15)

C:\Users\Chheten\AppData\Local\Temp\ipykernel_15076\280438611.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Selected_df_clean ['Predict_score']= Predict_score #new column and add the values from predict_score


,loan_amnt,int_rate,fico_range_low,dti,annual_inc,delinq_2yrs,Predict_score
0,3600.0,13.99,675.0,5.91,55000.0,0.0,16.769151
1,24700.0,11.99,715.0,16.06,65000.0,1.0,18.581265
2,20000.0,10.78,695.0,10.78,63000.0,0.0,14.887282
3,10400.0,22.45,695.0,25.37,104433.0,1.0,37.937304
4,11950.0,13.44,690.0,10.20,34000.0,0.0,19.025694
5,20000.0,9.17,680.0,14.67,180000.0,0.0,9.375239
6,20000.0,8.49,705.0,17.61,85000.0,1.0,11.361048
7,10000.0,6.49,685.0,13.07,85000.0,0.0,7.068730
8,8000.0,11.48,700.0,34.80,42000.0,0.0,17.841205
9,1400.0,12.88,700.0,34.95,64000.0,0.0,17.705716


In [142]:
#saving new cleaned data into sql db, new db called borrowers_with_pd
Selected_df_clean.to_sql('borrowers_with_pd', connect, if_exists='replace', index=False)
print("saved")

saved


In [146]:
# Export all 3 tables to CSV files
import pandas as pd

# Read each table from SQL
borrowers_data = pd.read_sql("SELECT * FROM borrowers", connect)
risk_data = pd.read_sql("SELECT * FROM risk_data", connect)
borrowers_with_pd = pd.read_sql("SELECT * FROM borrowers_with_pd", connect)

# Save as CSV files
borrowers_data.to_csv('borrowers.csv', index=False)
risk_data.to_csv('risk_data.csv', index=False)
borrowers_with_pd.to_csv('borrowers_with_pd.csv', index=False)

print(" CVS Created ")

✓ CSV files created!


In [148]:
# Add risk_level column based on FICO scores
def assign_risk_level(fico):
    if fico < 680:
        return 'HIGH'
    elif fico >= 680 and fico < 720:
        return 'MEDIUM'
    else:
        return 'LOW'

# Add the column
Selected_df_clean['risk_level'] = Selected_df_clean['fico_range_low'].apply(assign_risk_level)

# Re-export to CSV
Selected_df_clean.to_csv('borrowers_with_pd.csv', index=False)

print(" Updated ")

C:\Users\Chheten\AppData\Local\Temp\ipykernel_15076\2099619083.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Selected_df_clean['risk_level'] = Selected_df_clean['fico_range_low'].apply(assign_risk_level)


Updated 
